# Data Center LST Extraction — Run Notebook

Imports functions from `gee_functions.py` and runs the actual Earth Engine
exports. Keep this notebook in the same folder as `gee_functions.py`
(or adjust the path in the import cell below).

Each dataset is in its own cell so you can run/monitor/rerun them
independently rather than firing everything at once.

In [3]:
import ee
#ee.Authenticate()

Enter verification code:  4/1AXEQxIBN1RBD5ol7Uzs8JkcD0TVMlgpzMQFKoKZ0kt8y-Izaam3EP7uSVJY



Successfully saved authorization token.


In [5]:
import sys
import ee
sys.path.append('.') 

import gee_functions as dc
dc.initialize_ee('curious-setup-498812-j5')

In [ ]:
# Maps each asset to its sensor type so the loop knows which
# exports and scales to run for each one

ASSETS = {
    # 'modis_hyperscale_all':  'modis',
    # 'modis_iso10000m':       'modis',
    # 'modis_iso7500m':        'modis',
    # 'modis_iso5000m':        'modis',
    # 'modis_iso2500m':        'modis',
    'landsat_hyperscale_all': 'landsat',
    'landsat_iso10000m':      'landsat',
    'landsat_iso7500m':       'landsat',
    'landsat_iso5000m':       'landsat',
    'landsat_iso2500m':       'landsat',
}

RUN = {
    'modis_monthly_day':    False,
    'modis_monthly_night':  False,
    'modis_daily_obs':      False,
    'modis_nlcd':           False,
    'landsat_monthly_day':  False,
    'landsat_obs':          False,
    'landsat_nlcd':         False,
    'modis_monthly_best':   False,
    'landsat_monthly_best': False,
    'landsat_monthly_obs':  True,
}

MODIS_START,   MODIS_END   = 2001, 2025
LANDSAT_START, LANDSAT_END = 2013, 2025

TOTAL_BATCHES_MODIS          = 8   # monthly MODIS (relatively small)
TOTAL_BATCHES_MODIS_DAILY    = 40  # daily MODIS (~365x more images per year)
TOTAL_BATCHES_LANDSAT        = 80  # individual Landsat scenes
TOTAL_BATCHES_LANDSAT_MONTHLY = 20 # monthly aggregated Landsat

In [4]:
upload_tasks = dc.upload_shapefiles_as_assets(
    shp_dir="../data/data_raw/shp",
    asset_root="projects/curious-setup-498812-j5/assets"
)
dc.wait_for_tasks(upload_tasks)

Deleted existing asset: modis_hyperscale_all
Queued asset upload: modis_hyperscale_all
Deleted existing asset: landsat_iso7500m
Queued asset upload: landsat_iso7500m
Deleted existing asset: modis_iso2500m
Queued asset upload: modis_iso2500m
Deleted existing asset: landsat_iso2500m
Queued asset upload: landsat_iso2500m
Deleted existing asset: modis_iso10000m
Queued asset upload: modis_iso10000m
Deleted existing asset: modis_iso5000m
Queued asset upload: modis_iso5000m
Deleted existing asset: landsat_hs_extra
Queued asset upload: landsat_hs_extra
Deleted existing asset: landsat_all146
Queued asset upload: landsat_all146
Deleted existing asset: landsat_hyperscale_all
Queued asset upload: landsat_hyperscale_all
Deleted existing asset: landsat_pre2014
Queued asset upload: landsat_pre2014
Deleted existing asset: landsat_iso5000m
Queued asset upload: landsat_iso5000m
Deleted existing asset: landsat_iso10000m
Queued asset upload: landsat_iso10000m
Deleted existing asset: modis_iso7500m
Queued 

## base variables

In [4]:
all_tasks = []

for asset_id, sensor in ASSETS.items():
    print(f"\n{'='*50}")
    print(f"Queuing exports for: {asset_id} ({sensor})")
    print(f"{'='*50}")

    folder = f'EarthEngine_Exports_{asset_id}'
    dc_list, total_size = dc.get_base_variables(asset_id=asset_id)
    n_dcs = total_size.getInfo()
    print(f"  {n_dcs} data centers loaded")
    
    import math
    def non_empty_batches(total_setting, n):
        # Mirrors GEE's server-side arithmetic: batch_size = ceil(n / total),
        # then the number of non-empty slots = ceil(n / batch_size).
        # Prevents wasted tasks writing 2-byte empty CSVs for tail slots.
        total    = min(total_setting, n)
        batch_sz = math.ceil(n / total)
        return math.ceil(n / batch_sz)

    actual_modis_batches           = non_empty_batches(TOTAL_BATCHES_MODIS, n_dcs)
    actual_modis_daily_batches     = non_empty_batches(TOTAL_BATCHES_MODIS_DAILY, n_dcs)
    actual_landsat_batches         = non_empty_batches(TOTAL_BATCHES_LANDSAT, n_dcs)
    actual_landsat_monthly_batches = non_empty_batches(TOTAL_BATCHES_LANDSAT_MONTHLY, n_dcs)

    if sensor == 'modis':
        if RUN['modis_monthly_day']:
            monthly_day = dc.generate_monthly_clearsky_mean(
                MODIS_START, MODIS_END, period='day')
            for i in range(actual_modis_batches):
                t = dc.extract_and_export(
                    i, actual_modis_batches, dc_list, total_size,
                    monthly_day, f"{asset_id}_MODIS_Monthly_Day",
                    scale=1000, drop_nulls=False, drive_folder=folder,
                    skip_existing=True)
                if t: all_tasks.append(t)

        if RUN['modis_monthly_night']:
            monthly_night = dc.generate_monthly_clearsky_mean(
                MODIS_START, MODIS_END, period='night')
            for i in range(actual_modis_batches):
                t = dc.extract_and_export(
                    i, actual_modis_batches, dc_list, total_size,
                    monthly_night, f"{asset_id}_MODIS_Monthly_Night",
                    scale=1000, drop_nulls=False, drive_folder=folder,
                    skip_existing=True)
                if t: all_tasks.append(t)

        if RUN['modis_daily_obs']:
            daily_modis = dc.generate_daily_observations(
                MODIS_START, MODIS_END)
            for i in range(actual_modis_daily_batches):
                t = dc.extract_and_export(
                    i, actual_modis_daily_batches, dc_list, total_size,
                    daily_modis, f"{asset_id}_MODIS_Daily_Obs",
                    scale=1000, drop_nulls=True, drive_folder=folder,
                    skip_existing=True)
                if t: all_tasks.append(t)

        if RUN['modis_nlcd']:
            nlcd = dc.generate_annual_nlcd(MODIS_START, MODIS_END, scale=1000)
            for i in range(actual_modis_batches):
                t = dc.extract_and_export(
                    i, actual_modis_batches, dc_list, total_size,
                    nlcd, f"{asset_id}_NLCD_1000m",
                    scale=1000, drop_nulls=False, drive_folder=folder,
                    skip_existing=True)
                if t: all_tasks.append(t)

        if RUN['modis_monthly_best']:
            monthly_best = dc.generate_monthly_best_day(
                MODIS_START, MODIS_END, period='day')
            for i in range(actual_modis_batches):
                t = dc.extract_and_export(
                    i, actual_modis_batches, dc_list, total_size,
                    monthly_best, f"{asset_id}_MODIS_Monthly_Best",
                    scale=1000, drop_nulls=False, drive_folder=folder,
                    skip_existing=True)
                if t: all_tasks.append(t)

    elif sensor == 'landsat':
        if RUN['landsat_monthly_day']:
            # Landsat is daytime only — no nighttime thermal equivalent.
            landsat_monthly = dc.generate_monthly_landsat_mean(
                LANDSAT_START, LANDSAT_END)
            for i in range(actual_landsat_monthly_batches):
                t = dc.extract_and_export(
                    i, actual_landsat_monthly_batches, dc_list, total_size,
                    landsat_monthly, f"{asset_id}_Landsat_Monthly_Day",
                    scale=150, drop_nulls=False, drive_folder=folder,
                    skip_existing=True)
                if t: all_tasks.append(t)

        if RUN['landsat_obs']:
            landsat_obs = dc.generate_all_landsat_observations(
                LANDSAT_START, LANDSAT_END)
            for i in range(actual_landsat_batches):
                t = dc.extract_and_export(
                    i, actual_landsat_batches, dc_list, total_size,
                    landsat_obs, f"{asset_id}_Landsat_Obs",
                    scale=150, drop_nulls=True, drive_folder=folder,
                    skip_existing=True)
                if t: all_tasks.append(t)

        if RUN['landsat_nlcd']:
            nlcd = dc.generate_annual_nlcd(LANDSAT_START, LANDSAT_END, scale=150)
            for i in range(actual_landsat_batches):
                t = dc.extract_and_export(
                    i, actual_landsat_batches, dc_list, total_size,
                    nlcd, f"{asset_id}_NLCD_150m",
                    scale=150, drop_nulls=False, drive_folder=folder,
                    skip_existing=True)
                if t: all_tasks.append(t)

        if RUN['landsat_monthly_best']:
            landsat_best = dc.generate_monthly_best_landsat(
                LANDSAT_START, LANDSAT_END)
            for i in range(actual_landsat_monthly_batches):
                t = dc.extract_and_export(
                    i, actual_landsat_monthly_batches, dc_list, total_size,
                    landsat_best, f"{asset_id}_Landsat_Monthly_Best",
                    scale=150, drop_nulls=False, drive_folder=folder,
                    skip_existing=True)
                if t: all_tasks.append(t)

        if RUN['landsat_monthly_obs']:
            obs_batch_count = n_dcs
            for i in range(obs_batch_count):
                # Compute the geographic extent of this batch's DC locations.
                # filterBounds inside generate_monthly_obs_landsat then restricts
                # scene candidates to WRS-2 tiles that actually cover these DCs.
                # buffer(10000) expands beyond the 7500m DC buffer radius so scenes
                # whose footprint just clips the buffer edge are not missed.
                batch_sz  = total_size.divide(obs_batch_count).ceil()
                batch_aoi = ee.FeatureCollection(
                    dc_list.slice(batch_sz.multiply(i), batch_sz.multiply(i + 1))
                ).geometry().bounds().buffer(10000)
        
                monthly_obs = dc.generate_monthly_obs_landsat(
                    LANDSAT_START, LANDSAT_END, window_days=14, aoi=batch_aoi
                )
                t = dc.extract_and_export(
                    i, obs_batch_count, dc_list, total_size,
                    monthly_obs, f"{asset_id}_Landsat_Monthly_Obs",
                    scale=150, drop_nulls=True, drive_folder=folder,
                    skip_existing=False)
                if t: all_tasks.append(t)

print(f"\nTotal tasks queued: {len(all_tasks)}")


Queuing exports for: landsat_hyperscale_all (landsat)
  10 data centers loaded
Queued: landsat_hyperscale_all_Landsat_Monthly_Obs_Batch_1 @ scale=150m -> drive
Queued: landsat_hyperscale_all_Landsat_Monthly_Obs_Batch_2 @ scale=150m -> drive
Queued: landsat_hyperscale_all_Landsat_Monthly_Obs_Batch_3 @ scale=150m -> drive
Queued: landsat_hyperscale_all_Landsat_Monthly_Obs_Batch_4 @ scale=150m -> drive
Queued: landsat_hyperscale_all_Landsat_Monthly_Obs_Batch_5 @ scale=150m -> drive
Queued: landsat_hyperscale_all_Landsat_Monthly_Obs_Batch_6 @ scale=150m -> drive
Queued: landsat_hyperscale_all_Landsat_Monthly_Obs_Batch_7 @ scale=150m -> drive
Queued: landsat_hyperscale_all_Landsat_Monthly_Obs_Batch_8 @ scale=150m -> drive
Queued: landsat_hyperscale_all_Landsat_Monthly_Obs_Batch_9 @ scale=150m -> drive
Queued: landsat_hyperscale_all_Landsat_Monthly_Obs_Batch_10 @ scale=150m -> drive

Queuing exports for: landsat_iso10000m (landsat)
  30 data centers loaded
Queued: landsat_iso10000m_Landsat_

In [5]:
import math

LANDSAT_START, LANDSAT_END = 2016, 2025

# Hyperscale export_ids first (your priority sites; also the pilot)
HYPERSCALE_IDS = [411, 412, 648, 664, 2949, 2950, 2998, 3012, 3051, 3052]

PILOT_ONLY = False          # <- flip to False after checking pilot EECU cost

dc_list, total_size = dc.get_base_variables('landsat_all146')
n_dcs = total_size.getInfo()
print(f"{n_dcs} DCs in asset")

# Build an index order: hyperscale batches first, then the rest
all_feats = [ee.Feature(dc_list.get(i)) for i in range(n_dcs)]
eids      = [f.get('export_id').getInfo() for f in all_feats]
order     = ([i for i, e in enumerate(eids) if e in HYPERSCALE_IDS] +
             [i for i, e in enumerate(eids) if e not in HYPERSCALE_IDS])

run_indices = order[:len(HYPERSCALE_IDS)] if PILOT_ONLY else order
folder = 'EarthEngine_Exports_landsat_all146'
all_tasks = []

for i in run_indices:
    f   = all_feats[i]
    aoi = f.geometry().buffer(5000)   # scene-candidate AOI (1500m buffer + margin)

    obs_30 = dc.generate_all_landsat_obs_30m(LANDSAT_START, LANDSAT_END, aoi=aoi)
    # one DC per batch: pass a single-feature list
    single_list = ee.List([f])
    t = dc.extract_and_export(
        0, 1, single_list, ee.Number(1),
        obs_30, f"landsat_all146_Obs30m_DC{eids[i]}",
        scale=30, drop_nulls=True, buffer_m=1500,
        drive_folder=folder, skip_existing=True)
    if t: all_tasks.append(t)

print(f"Queued: {len(all_tasks)} tasks ({'PILOT' if PILOT_ONLY else 'FULL'})")

146 DCs in asset
Skipping landsat_all146_Obs30m_DC664_Batch_1 - completed
Skipping landsat_all146_Obs30m_DC3012_Batch_1 - completed
Skipping landsat_all146_Obs30m_DC2950_Batch_1 - completed
Skipping landsat_all146_Obs30m_DC2949_Batch_1 - completed
Skipping landsat_all146_Obs30m_DC3052_Batch_1 - completed
Skipping landsat_all146_Obs30m_DC412_Batch_1 - completed
Skipping landsat_all146_Obs30m_DC3051_Batch_1 - completed
Skipping landsat_all146_Obs30m_DC648_Batch_1 - completed
Skipping landsat_all146_Obs30m_DC2998_Batch_1 - completed
Skipping landsat_all146_Obs30m_DC411_Batch_1 - completed
Queued: landsat_all146_Obs30m_DC120_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_Obs30m_DC1623_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_Obs30m_DC2444_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_Obs30m_DC1975_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_Obs30m_DC42_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_Obs30m_DC2709_Batch_1 @ scale=30m -> drive
Queued: landsa

In [4]:
nlcd_30 = dc.generate_annual_nlcd(LANDSAT_START, LANDSAT_END, scale=30)
n_batches = 20
for i in range(n_batches):
    t = dc.extract_and_export(
        i, n_batches, dc_list, total_size,
        nlcd_30, "landsat_all146_NLCD_30m",
        scale=30, drop_nulls=False, buffer_m=1500,
        drive_folder=folder, skip_existing=True)
    if t: all_tasks.append(t)

Queued: landsat_all146_NLCD_30m_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_2 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_3 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_4 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_5 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_6 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_7 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_8 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_9 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_10 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_11 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_12 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_13 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_14 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_15 @ scale=30m -> drive
Queued: landsat_all146_NLCD_30m_Batch_16 @ scale=30m -> drive
Queued: landsat_a

In [5]:
HYPERSCALE_IDS = [411, 412, 648, 664, 2949, 2950, 2998, 3012, 3051, 3052]

dc_list, total_size = dc.get_base_variables('landsat_all146')
n_dcs = total_size.getInfo()
all_feats = [ee.Feature(dc_list.get(i)) for i in range(n_dcs)]
eids      = [f.get('export_id').getInfo() for f in all_feats]

folder = 'EarthEngine_Exports_landsat_all146'
l5_tasks = []
for i in [ix for ix, e in enumerate(eids) if e in HYPERSCALE_IDS]:
    f, aoi = all_feats[i], all_feats[i].geometry().buffer(5000)
    t = dc.extract_and_export(0, 1, ee.List([f]), ee.Number(1),
        dc.generate_all_l5_obs_30m(1993, 2011, aoi=aoi),
        f"landsat_all146_L5Obs30m_DC{eids[i]}",
        scale=30, drop_nulls=True, buffer_m=1500,
        drive_folder=folder, skip_existing=True)
    if t: l5_tasks.append(t)
print(f"Queued: {len(l5_tasks)} L5 hyperscale tasks")   # expect 10

Queued: landsat_all146_L5Obs30m_DC664_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC3012_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC2950_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC2949_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC3052_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC412_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC3051_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC648_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC2998_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC411_Batch_1 @ scale=30m -> drive
Queued: 10 L5 hyperscale tasks


In [6]:
dc_list, total_size = dc.get_base_variables('landsat_hs_extra')
n = total_size.getInfo()
feats = [ee.Feature(dc_list.get(i)) for i in range(n)]
eids  = [f.get('export_id').getInfo() for f in feats]
ALREADY = [411,412,648,664,2949,2950,2998,3012,3051,3052]

folder = 'EarthEngine_Exports_landsat_hs_extra'
tasks = []
for i in [ix for ix, e in enumerate(eids) if e not in ALREADY]:
    f, aoi = feats[i], feats[i].geometry().buffer(5000)
    t = dc.extract_and_export(0, 1, ee.List([f]), ee.Number(1),
        dc.generate_all_landsat_obs_30m(2013, 2026, aoi=aoi),
        f"landsat_hs_extra_Obs30m_DC{eids[i]}",
        scale=30, drop_nulls=True, buffer_m=1500,
        drive_folder=folder, skip_existing=True)
    if t: tasks.append(t)
print(f"Queued: {len(tasks)} of {n} total ({n - len(tasks)} already covered)")

Queued: landsat_hs_extra_Obs30m_DC2751_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC2799_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC2070_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC414_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC3277_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC565_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC574_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC3014_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC473_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC2851_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC456_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC2060_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC2808_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC3015_Batch_1 @ scale=30m -> drive
Queued: landsat_hs_extra_Obs30m_DC3004_Batch_1 @ scal

In [7]:
# L5 for ALL known-year hyperscale facilities, 30m, buffer 1500m.
# Original 10 -> landsat_all146 folder/naming; pre-2014 -> landsat_pre2014.
# skip_existing=True makes re-runs safe.

HS_146 = [411,412,648,664,2949,2950,2998,3012,3051,3052]

jobs = [
    # (asset, id_filter, l5_start)
    ('landsat_all146',  HS_146, 1995),   # openers 2014-2024: L5 pre-period only;
                                         # 2003 start = rel <= -11 for earliest opener
    ('landsat_pre2014', None,   1995),   # openers 2000-2013: pre + early post
]

all_l5_tasks = []
for asset, id_filter, l5_start in jobs:
    dc_list, total_size = dc.get_base_variables(asset)
    n = total_size.getInfo()
    feats = [ee.Feature(dc_list.get(i)) for i in range(n)]
    eids  = [f.get('export_id').getInfo() for f in feats]

    run_ix = (range(n) if id_filter is None
              else [ix for ix, e in enumerate(eids) if e in id_filter])
    folder = f'EarthEngine_Exports_{asset}'

    for i in run_ix:
        f, aoi = feats[i], feats[i].geometry().buffer(5000)
        t = dc.extract_and_export(0, 1, ee.List([f]), ee.Number(1),
            dc.generate_all_l5_obs_30m(l5_start, 2011, aoi=aoi),
            f"{asset}_L5Obs30m_DC{eids[i]}",
            scale=30, drop_nulls=True, buffer_m=1500,
            drive_folder=folder, skip_existing=False)
        if t: all_l5_tasks.append(t)

print(f"Queued: {len(all_l5_tasks)} L5 tasks (expect 10 + pre2014 count)")

Queued: landsat_all146_L5Obs30m_DC664_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC3012_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC2950_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC2949_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC3052_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC412_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC3051_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC648_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC2998_Batch_1 @ scale=30m -> drive
Queued: landsat_all146_L5Obs30m_DC411_Batch_1 @ scale=30m -> drive
Queued: landsat_pre2014_L5Obs30m_DC2039_Batch_1 @ scale=30m -> drive
Queued: landsat_pre2014_L5Obs30m_DC2886_Batch_1 @ scale=30m -> drive
Queued: landsat_pre2014_L5Obs30m_DC2899_Batch_1 @ scale=30m -> drive
Queued: landsat_pre2014_L5Obs30m_DC2988_Batch_1 @ scale=30m -> drive
Queued: landsat_pre2014_L5Obs30m_DC3009_Batch_1 